# Train HelmetGuard's models on Kaggle (version 2)

Trains the two detectors the Android app and the website need, and packages them.

**Set up, once, in this editor**
1. Under **Settings → Accelerator**, choose **GPU T4 x2**.
2. Under **Settings**, check that internet is on.
3. Optional, to retrain stage 2 as well: under **Add-ons → Secrets**, add a secret with the label `ROBOFLOW_API_KEY` and your free [Roboflow](https://app.roboflow.com) **Private API key** as the value, then tick it so this notebook can use it.

**To run it**

Click **Save Version**, choose **Save & Run All**, then **Save**. It runs in the background, so you can close the browser. Kaggle allows 12 hours and this run should take about 5.

**When it finishes**

Open the saved version, look at the sample pictures, then download **helmetguard_models.zip** from its **Output** section.

**What this run uses**

- *HELMET* by Siebert and Lin (2020), CC BY 4.0, https://osf.io/4pwj8/: 910 clips of road traffic in Myanmar with every nearby motorcycle boxed. About one frame per second is used.
- *Trafffic_Violations_Triple_Riding_No_Helmet_Plate_* by DevguruCodes on Kaggle, ODbL: Indian traffic pictures. Add it under **Add Input** if this notebook can't fetch it by itself.
- *helmet-detection-and-license-plate-recognition* by on Gia Capital, CC BY 4.0, via Roboflow Universe, **only if a Roboflow key is set up** (see below). It has the head labels, so without it stage 2 isn't retrained and the app keeps its current head model.
- A large COCO-trained YOLO11x only labels ridden motorcycles that the datasets left without a box. It is not shipped.

Number plates are never a training label, so the models can't detect them.

**Accuracy warning.** Neither dataset was filmed in Pakistan, so expect real results on your roads to be somewhat worse than these scores. Your own footage is still the best way to improve the model.


## 1. Set up


In [ ]:
%pip install -q ultralytics==8.4.208 onnx==1.23.2 onnxruntime==1.31.1 onnxslim==0.1.99


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Choose Settings > Accelerator > GPU T4 x2, then Save & Run All again.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
from pathlib import Path

from ultralytics import settings

# Kaggle has logging tools installed. Left on, one could stop a background run to ask for a login.
os.environ["WANDB_MODE"] = "disabled"
settings.update({"wandb": False, "mlflow": False, "comet": False, "clearml": False})

VERSION = "yolo11n-v2"
STAGE1_MODEL = "yolo11n.pt"
STAGE2_MODEL = "yolo11n.pt"
STAGE1_SIZE = 640  # was 416: small and distant bikes need the extra pixels
STAGE2_SIZE = 320
STAGE1_EPOCHS = 150
STAGE2_EPOCHS = 100
PATIENCE = 30
STAGE1_BATCH = 32
STAGE2_BATCH = 64
WORKERS = 4
CACHE = "ram"  # Ultralytics checks free memory first and skips the cache if it won't fit
DEVICE = 0
SEED = 42
STAGE2_NAMES = ["helmet", "no_helmet"]

# Fills in boxes for ridden bikes the datasets didn't label.
LABEL_HELPER = "yolo11x.pt"
LABEL_HELPER_SIZE = 1280

# HELMET: one frame every N frames (10 fps video). None means every clip.
HELMET_EVERY_TRAIN = 10
HELMET_EVERY_EVAL = 20
HELMET_LIMIT_CLIPS = None

# Where Kaggle puts datasets added under Add Input.
EXTRA_SEARCH = [Path("/kaggle/input")]

# Kaggle keeps everything in /kaggle/working as the run's output, so the images stay out of it.
SAVE_DIR = Path("/kaggle/working")
TEMP = Path("/kaggle/temp")
RUNS = SAVE_DIR / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)
ZIP = TEMP / "dataset_yolov8.zip"
RAW = TEMP / "raw"
PREPARED = TEMP / "prepared"
VAL_DIR = TEMP / "val"
print("Saving results to", SAVE_DIR)


## 2. Get and prepare the data


In [ ]:
from kaggle_secrets import UserSecretsClient


def read_api_key():
    # Optional. Without it, the Roboflow dataset is left out and stage 2 isn't retrained.
    try:
        return UserSecretsClient().get_secret("ROBOFLOW_API_KEY").strip() or None
    except Exception:
        return None


In [ ]:
import time
import zipfile

import requests

DATASET_URL = "https://api.roboflow.com/on-gia-capital/helmet-detection-and-license-plate-recognition/1/yolov8"


def export_link(api_key):
    # Roboflow answers 202 while it prepares the export. Errors never print the URL, which holds the key.
    for _ in range(120):
        try:
            response = requests.get(DATASET_URL, params={"api_key": api_key}, timeout=60)
        except requests.RequestException as error:
            raise RuntimeError(f"Couldn't reach Roboflow ({type(error).__name__}). Check that this notebook has internet access, then run it again.") from None
        if response.status_code == 200:
            return response.json()["export"]["link"]
        if response.status_code != 202:
            raise RuntimeError(f"Roboflow refused the request (HTTP {response.status_code}). Check your Private API key, then Run all again.")
        print("Roboflow is preparing the download...")
        time.sleep(10)
    raise RuntimeError("Roboflow took too long to prepare the download. Run all again in a few minutes.")


if not (RAW / "data.yaml").exists() and not ZIP.exists():
    key = read_api_key()
    if key:
        link = export_link(key)
        del key
        part = ZIP.with_suffix(".part")
        with requests.get(link, stream=True, timeout=120) as response:
            response.raise_for_status()
            with open(part, "wb") as file:
                for chunk in response.iter_content(chunk_size=1 << 20):
                    file.write(chunk)
        part.rename(ZIP)

if not (RAW / "data.yaml").exists() and ZIP.exists():
    RAW.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as archive:
        archive.extractall(RAW)

ROBOFLOW = (RAW / "data.yaml").exists()
if ROBOFLOW:
    print("Roboflow images:", sum(1 for _ in RAW.glob("*/images/*")))
else:
    print("No Roboflow key, so its dataset is left out and stage 2 won't be retrained.")


In [ ]:
%%writefile prepare_public_dataset.py
#!/usr/bin/env python3
"""Turn a Roboflow YOLOv8 export into HelmetGuard's two training sets.

Stage 1 finds whole motorcycles in the road scene. Stage 2 looks inside one
motorcycle crop for bare heads and helmets. In the public dataset the "Rider"
box already wraps the bike and everyone on it, so it becomes the stage 1 label
and the crop for stage 2.

The dataset also boxes number plates. HelmetGuard doesn't detect plates, so they
never become a training label; they are only used to notice a bike that the
dataset forgot to box.

The original train/valid/test split is thrown away. Frames from one video look
almost identical, so a random split would inflate the scores; here every frame
of a video stays together in one split.
"""
from __future__ import annotations

import argparse
import ast
import hashlib
import json
import re
import shutil
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

from PIL import Image, ImageOps

RAW_SPLITS = ("train", "valid", "test")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
CROP_PADDING = 0.08
MIN_CROP_HEIGHT_PX = 32
MIN_PART_SIZE_PX = 3
STAGE2_NAMES = ["helmet", "no_helmet"]
# Images the first model's stricter rule would also have kept, one name per line.
FIRST_RULE_FILE = "first_model_rule_images.txt"
# Images kept despite a head outside every bike box: {image name: [[cx, cy, w, h], ...]}.
UNBOXED_FILE = "unboxed_heads.json"


@dataclass(frozen=True)
class Box:
    """A box in 0..1 image coordinates."""

    kind: str
    cx: float
    cy: float
    w: float
    h: float

    @property
    def left(self) -> float:
        return self.cx - self.w / 2

    @property
    def right(self) -> float:
        return self.cx + self.w / 2

    @property
    def top(self) -> float:
        return self.cy - self.h / 2

    @property
    def bottom(self) -> float:
        return self.cy + self.h / 2

    def contains_center_of(self, other: "Box") -> bool:
        return self.left <= other.cx <= self.right and self.top <= other.cy <= self.bottom


def canonical_class(name: str) -> str | None:
    key = re.sub(r"[\s_\-]+", "", name.lower())
    return {
        "rider": "rider",
        "motorcyclist": "rider",
        "helmet": "helmet",
        "nohelmet": "no_helmet",
        "numberplate": "plate",
        "licenseplate": "plate",
        "plate": "plate",
    }.get(key)


def read_class_names(yaml_path: Path) -> list[str]:
    """Reads names from data.yaml in either list form Roboflow writes."""
    lines = yaml_path.read_text(encoding="utf-8").splitlines()
    for index, line in enumerate(lines):
        if not line.startswith("names:"):
            continue
        inline = line.split(":", 1)[1].strip()
        if inline:
            return [str(n) for n in ast.literal_eval(inline)]
        names: list[str] = []
        for entry in lines[index + 1 :]:
            match = re.match(r"\s+(?:-\s*|\d+:\s*)(.+)$", entry)
            if not match:
                break
            names.append(match.group(1).strip().strip("'\""))
        return names
    raise ValueError(f"No names found in {yaml_path}")


def read_labels(path: Path, names: list[str]) -> list[Box]:
    if not path.exists():
        return []
    boxes = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        kind = canonical_class(names[int(parts[0])])
        if kind is not None:
            boxes.append(Box(kind, *(float(v) for v in parts[1:])))
    return boxes


def group_key(filename: str) -> str:
    """Frames of one video share a key. Unrelated photos each get their own."""
    stem = Path(filename).name.split(".rf.")[0]
    if not re.search(r"(?i)mov|mp4|avi", stem):
        return stem
    stem = re.sub(r"(?i)_(jpg|jpeg|png)$", "", stem)
    return re.sub(r"[-_]*\d*[-_]*$", "", stem)


def assign_split(key: str) -> str:
    bucket = int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) % 100
    if bucket < 70:
        return "train"
    return "val" if bucket < 85 else "test"


def owner_of(part: Box, riders: list[Box]) -> Box | None:
    """Where bikes overlap, a part belongs to the tightest bike around it."""
    candidates = [r for r in riders if r.contains_center_of(part)]
    return min(candidates, key=lambda r: r.w * r.h) if candidates else None


def write_yaml(root: Path, names: list[str]) -> Path:
    body = [f"path: {root.resolve().as_posix()}", "train: images/train", "val: images/val", "test: images/test", "names:"]
    body += [f"  {index}: {name}" for index, name in enumerate(names)]
    path = root / "data.yaml"
    path.write_text("\n".join(body) + "\n", encoding="utf-8")
    return path


def make_dirs(root: Path) -> None:
    for split in ("train", "val", "test"):
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)


def crop_rider(image: Image.Image, rider: Box, parts: list[Box]):
    """Returns the padded crop and its YOLO label lines, or None if unusable."""
    width, height = image.size
    pad_x = rider.w * CROP_PADDING
    pad_y = rider.h * CROP_PADDING
    left = max(0, int((rider.left - pad_x) * width))
    top = max(0, int((rider.top - pad_y) * height))
    right = min(width, int((rider.right + pad_x) * width) + 1)
    bottom = min(height, int((rider.bottom + pad_y) * height) + 1)
    crop_w = right - left
    crop_h = bottom - top
    if crop_h < MIN_CROP_HEIGHT_PX or crop_w < MIN_CROP_HEIGHT_PX / 2:
        return None

    lines = []
    for part in parts:
        if part.kind not in STAGE2_NAMES:
            continue
        x1 = max(part.left * width, left)
        y1 = max(part.top * height, top)
        x2 = min(part.right * width, right)
        y2 = min(part.bottom * height, bottom)
        if x2 - x1 < MIN_PART_SIZE_PX or y2 - y1 < MIN_PART_SIZE_PX:
            continue
        cx = ((x1 + x2) / 2 - left) / crop_w
        cy = ((y1 + y2) / 2 - top) / crop_h
        lines.append(f"{STAGE2_NAMES.index(part.kind)} {cx:.6f} {cy:.6f} {(x2 - x1) / crop_w:.6f} {(y2 - y1) / crop_h:.6f}")
    if not lines:
        return None
    return image.crop((left, top, right, bottom)), lines


def prepare(raw: Path, out: Path, keep_unboxed: bool = False) -> dict:
    """Builds stage1/ and stage2/ under `out`.

    With keep_unboxed, images with a head outside every bike box stay in stage 1 and are
    listed in stage1/UNBOXED_FILE, for pseudo_labels.drop_uncovered() to settle later.
    """
    names = read_class_names(raw / "data.yaml")
    missing = {"rider", "helmet", "no_helmet", "plate"} - {canonical_class(n) for n in names}
    if missing:
        raise ValueError(f"Dataset is missing classes: {sorted(missing)}. It has: {names}")

    stage1 = out / "stage1"
    stage2 = out / "stage2"
    make_dirs(stage1)
    make_dirs(stage2)

    stats: Counter = Counter()
    seen_groups: dict[str, set] = {"train": set(), "val": set(), "test": set()}
    first_rule: list[str] = []
    unboxed: dict[str, list[list[float]]] = {}

    for raw_split in RAW_SPLITS:
        image_dir = raw / raw_split / "images"
        if not image_dir.exists():
            continue
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            boxes = read_labels(raw / raw_split / "labels" / f"{image_path.stem}.txt", names)
            riders = [b for b in boxes if b.kind == "rider"]
            parts = [b for b in boxes if b.kind != "rider"]
            key = group_key(image_path.name)
            split = assign_split(key)
            seen_groups[split].add(key)
            stats["images_total"] += 1

            owners = {part: owner_of(part, riders) for part in parts}

            # A head outside every rider box means a bike nobody boxed. Teaching stage 1
            # that this bike is background would hurt it. Plates don't count: the dataset
            # also boxes the plates of cars, which are rightly outside every bike box.
            unowned_heads = [part for part, owner in owners.items() if owner is None and part.kind in STAGE2_NAMES]
            if unowned_heads and not keep_unboxed:
                stats["stage1_skipped_unboxed_bike"] += 1
            else:
                if unowned_heads:
                    # Kept for now: pseudo_labels.drop_uncovered() removes it again unless
                    # the filled-in boxes end up covering every one of these heads.
                    unboxed[image_path.name] = [[h.cx, h.cy, h.w, h.h] for h in unowned_heads]
                    stats["stage1_unboxed_bike_images_kept_for_now"] += 1
                shutil.copy2(image_path, stage1 / "images" / split / image_path.name)
                lines = [f"0 {r.cx:.6f} {r.cy:.6f} {r.w:.6f} {r.h:.6f}" for r in riders]
                (stage1 / "labels" / split / f"{image_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
                stats[f"stage1_images_{split}"] += 1
                stats[f"stage1_boxes_{split}"] += len(riders)
                # The first model also left out images with an unboxed plate. Note which images it
                # would have kept, so its test score can be compared on exactly the same pictures.
                if any(owner is None for owner in owners.values()):
                    if not unowned_heads:
                        stats["stage1_kept_with_unboxed_plate"] += 1
                else:
                    first_rule.append(image_path.name)

            if not riders:
                continue
            with Image.open(image_path) as opened:
                image = ImageOps.exif_transpose(opened).convert("RGB")
                for index, rider in enumerate(riders):
                    own = [part for part, owner in owners.items() if owner == rider]
                    made = crop_rider(image, rider, own)
                    if made is None:
                        stats["stage2_crops_skipped"] += 1
                        continue
                    crop, lines = made
                    crop_name = f"{image_path.stem}__{index}"
                    crop.save(stage2 / "images" / split / f"{crop_name}.jpg", quality=95)
                    (stage2 / "labels" / split / f"{crop_name}.txt").write_text("\n".join(lines), encoding="utf-8")
                    stats[f"stage2_crops_{split}"] += 1
                    for line in lines:
                        stats[f"stage2_{STAGE2_NAMES[int(line.split()[0])]}_{split}"] += 1

    for split in ("train", "val", "test"):
        stats[f"source_groups_{split}"] = len(seen_groups[split])
        if stats[f"stage1_images_{split}"] == 0 or stats[f"stage2_crops_{split}"] == 0:
            raise ValueError(f"Split '{split}' came out empty: {dict(stats)}")

    (stage1 / FIRST_RULE_FILE).write_text("\n".join(first_rule), encoding="utf-8")
    (stage1 / UNBOXED_FILE).write_text(json.dumps(unboxed), encoding="utf-8")
    result = dict(sorted(stats.items()))
    result["stage1_yaml"] = str(write_yaml(stage1, ["motorcycle_unit"]))
    result["stage2_yaml"] = str(write_yaml(stage2, STAGE2_NAMES))
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--raw", type=Path, required=True, help="Folder holding data.yaml and train/valid/test")
    parser.add_argument("--out", type=Path, required=True, help="Where to write stage1/ and stage2/")
    args = parser.parse_args()
    print(json.dumps(prepare(args.raw, args.out), indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile helmet_dataset.py
#!/usr/bin/env python3
"""Adds frames from the HELMET dataset (Myanmar road traffic) to the stage 1 training set.

HELMET: Siebert and Lin, "Detecting motorcycle helmet use with deep learning",
Accident Analysis & Prevention 134 (2020). https://osf.io/4pwj8/ CC BY 4.0.

Its 910 clips are 10 seconds of 1920x1080 video at 10 fps, and every motorcycle
has a box around the bike and its riders, like this project's `motorcycle_unit`.
Neighbouring frames are almost identical, so only a few frames per clip are used,
and they are read straight out of the 29 GB of zip files with HTTP range requests
instead of downloading everything.

The dataset's own split (by clip) is kept, so no clip is in two splits.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import struct
import threading
import time
import urllib.error
import urllib.request
import zipfile
import zlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from PIL import Image

OSF = "https://osf.io/download/{}/"
ANNOTATIONS = "buh57"
SPLITS = "q7rmb"
IMAGE_PARTS = ["452nq", "muzgb", "9dmw3", "39ynb", "v3rch", "jyg9s", "6h5ka"]
SOURCE_SIZE = (1920, 1080)
SPLIT_NAMES = {"training": "train", "validation": "val", "test": "test"}
MIN_BOX_PX = 6
USER_AGENT = "HelmetGuard-training/1.0"


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


_no_redirect = urllib.request.build_opener(_NoRedirect)


def _request(url: str, headers: dict | None = None) -> urllib.request.Request:
    return urllib.request.Request(url, headers={"User-Agent": USER_AGENT, **(headers or {})})


def download(file_id: str, attempts: int = 5) -> bytes:
    """Downloads a small OSF file whole."""
    for attempt in range(attempts):
        try:
            with urllib.request.urlopen(_request(OSF.format(file_id)), timeout=300) as response:
                return response.read()
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == attempts - 1:
                raise
            time.sleep(5 * (attempt + 1))
    raise AssertionError("unreachable")


class OsfFile:
    """Random access to one OSF file.

    OSF redirects to a storage link that only works for about a minute, so the
    link is fetched again whenever it is older than LINK_AGE_S.
    """

    LINK_AGE_S = 40

    def __init__(self, file_id: str):
        self.file_id = file_id
        self._link = None
        self._link_time = 0.0
        self._lock = threading.Lock()
        self.size = int(self._get_raw(0, 0)[1].rsplit("/", 1)[-1])

    def _fresh_link(self, force: bool = False) -> str:
        with self._lock:
            if not force and self._link and time.time() - self._link_time < self.LINK_AGE_S:
                return self._link
            url = OSF.format(self.file_id)
            for _ in range(6):
                try:
                    with _no_redirect.open(_request(url), timeout=60):
                        break  # Answered without a redirect: use it as is.
                except urllib.error.HTTPError as error:
                    if error.code not in (301, 302, 303, 307, 308):
                        raise
                    url = error.headers["Location"]
                    if "osf.io" not in url.split("/")[2]:
                        break
            self._link, self._link_time = url, time.time()
            return url

    def _get_raw(self, start: int, end: int) -> tuple[bytes, str]:
        for attempt in range(6):
            link = self._fresh_link(force=attempt > 0)
            try:
                with urllib.request.urlopen(_request(link, {"Range": f"bytes={start}-{end}"}), timeout=120) as response:
                    if response.status != 206:
                        raise RuntimeError(f"Storage ignored the byte range (HTTP {response.status})")
                    return response.read(), response.headers.get("Content-Range", "")
            except (urllib.error.URLError, TimeoutError, ConnectionError):
                time.sleep(2 * (attempt + 1))
        raise RuntimeError(f"Couldn't read bytes {start}-{end} of OSF file {self.file_id}")

    def get(self, start: int, end: int) -> bytes:
        return self._get_raw(start, min(end, self.size - 1))[0]


class _Reader(io.RawIOBase):
    """Lets zipfile read the archive's table of contents from an OsfFile."""

    def __init__(self, remote: OsfFile):
        self.remote = remote
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.remote.size + offset
        return self.pos

    def readinto(self, buffer):
        if self.pos >= self.remote.size:
            return 0
        data = self.remote.get(self.pos, self.pos + len(buffer) - 1)
        buffer[: len(data)] = data
        self.pos += len(data)
        return len(data)


def list_members(remote: OsfFile) -> list[zipfile.ZipInfo]:
    with zipfile.ZipFile(io.BufferedReader(_Reader(remote), buffer_size=1 << 20)) as archive:
        return archive.infolist()


def read_member(remote: OsfFile, info: zipfile.ZipInfo) -> bytes:
    """One range request per file: the local header, then the data."""
    slack = 30 + len(info.filename.encode("utf-8")) + 1024
    blob = remote.get(info.header_offset, info.header_offset + slack + info.compress_size)
    signature, *_, name_len, extra_len = struct.unpack("<4sHHHHHIIIHH", blob[:30])
    if signature != b"PK\x03\x04":
        raise ValueError(f"Bad zip entry for {info.filename}")
    start = 30 + name_len + extra_len
    if start + info.compress_size > len(blob):
        blob = remote.get(info.header_offset, info.header_offset + start + info.compress_size)
    data = blob[start : start + info.compress_size]
    if info.compress_type == zipfile.ZIP_STORED:
        return data
    if info.compress_type == zipfile.ZIP_DEFLATED:
        return zlib.decompress(data, -15)
    raise ValueError(f"Unsupported compression {info.compress_type} for {info.filename}")


def clip_and_frame(name: str) -> tuple[str, int] | None:
    """'part_7/Mandalay_1_28/05.jpg' -> ('Mandalay_1_28', 5)."""
    path = Path(name)
    if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        return None
    digits = "".join(ch for ch in path.stem if ch.isdigit())
    if not digits:
        return None
    return path.parent.name, int(digits)


def read_annotations(blob: bytes) -> dict[str, dict[int, list[tuple[float, float, float, float]]]]:
    """{clip: {frame: [(x, y, w, h) in source pixels]}}"""
    boxes: dict[str, dict[int, list]] = defaultdict(lambda: defaultdict(list))
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        for name in archive.namelist():
            if not name.lower().endswith(".csv"):
                continue
            clip = Path(name).stem
            rows = csv.DictReader(io.StringIO(archive.read(name).decode("utf-8-sig")))
            for row in rows:
                boxes[clip][int(row["frame_id"])].append(
                    (float(row["x"]), float(row["y"]), float(row["w"]), float(row["h"]))
                )
    return boxes


def read_splits(blob: bytes) -> dict[str, str]:
    rows = csv.DictReader(io.StringIO(blob.decode("utf-8-sig")))
    return {row["video_id"].strip(): SPLIT_NAMES[row["Set"].strip().lower()] for row in rows}


def yolo_lines(boxes, width: int, height: int) -> list[str]:
    lines = []
    for x, y, w, h in boxes:
        x1, y1 = max(0.0, x), max(0.0, y)
        x2, y2 = min(float(width), x + w), min(float(height), y + h)
        if x2 - x1 < MIN_BOX_PX or y2 - y1 < MIN_BOX_PX:
            continue
        lines.append(
            f"0 {(x1 + x2) / 2 / width:.6f} {(y1 + y2) / 2 / height:.6f} {(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}"
        )
    return lines


def pick_frames(frames_with_boxes: set[int], available: set[int], every: int) -> list[int]:
    """Every `every`-th frame that has a file, preferring frames that show at least one bike."""
    picked = []
    for start in range(1, 101, every):
        window = [f for f in range(start, start + every) if f in available]
        if not window:
            continue
        busy = [f for f in window if f in frames_with_boxes]
        picked.append(busy[0] if busy else window[0])
    return picked


def add_helmet_frames(stage1: Path, every_train: int = 10, every_eval: int = 20, long_side: int = 640,
                      workers: int = 16, limit_clips: int | None = None, log=print) -> dict:
    """Writes frames into stage1/images/<split> and labels into stage1/labels/<split>."""
    started = time.time()
    annotations = read_annotations(download(ANNOTATIONS))
    splits = read_splits(download(SPLITS))
    stats: Counter = Counter()

    jobs = []
    for part in IMAGE_PARTS:
        remote = OsfFile(part)
        members = {}
        for info in list_members(remote):
            parsed = clip_and_frame(info.filename)
            if parsed:
                members.setdefault(parsed[0], {})[parsed[1]] = info
        log(f"HELMET {part}: {len(members)} clips")
        for clip, frames in sorted(members.items()):
            split = splits.get(clip)
            if split is None or clip not in annotations:
                stats["clips_without_split_or_labels"] += 1
                continue
            every = every_train if split == "train" else every_eval
            for frame in pick_frames(set(annotations[clip]), set(frames), every):
                jobs.append((remote, frames[frame], clip, frame, split))
            stats[f"clips_{split}"] += 1
            if limit_clips and sum(v for k, v in stats.items() if k.startswith("clips_")) >= limit_clips:
                break
        if limit_clips and sum(v for k, v in stats.items() if k.startswith("clips_")) >= limit_clips:
            break

    def work(job):
        remote, info, clip, frame, split = job
        try:
            data = read_member(remote, info)
            with Image.open(io.BytesIO(data)) as opened:
                image = opened.convert("RGB")
            width, height = image.size
            sx, sy = width / SOURCE_SIZE[0], height / SOURCE_SIZE[1]
            boxes = [(x * sx, y * sy, w * sx, h * sy) for x, y, w, h in annotations[clip].get(frame, [])]
            scale = long_side / max(width, height)
            if scale < 1:
                image = image.resize((round(width * scale), round(height * scale)), Image.BILINEAR)
                boxes = [(x * scale, y * scale, w * scale, h * scale) for x, y, w, h in boxes]
            stem = f"helmet_{clip}_{frame:03d}"
            image.save(stage1 / "images" / split / f"{stem}.jpg", quality=90)
            lines = yolo_lines(boxes, image.width, image.height)
            (stage1 / "labels" / split / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
            return split, len(lines), None
        except Exception as error:  # One bad frame shouldn't stop the run.
            return split, 0, f"{clip}/{frame}: {type(error).__name__}: {error}"

    for split in ("train", "val", "test"):
        (stage1 / "images" / split).mkdir(parents=True, exist_ok=True)
        (stage1 / "labels" / split).mkdir(parents=True, exist_ok=True)

    log(f"HELMET: reading {len(jobs)} frames")
    errors = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for done, (split, box_count, error) in enumerate(pool.map(work, jobs), 1):
            if error:
                errors.append(error)
                continue
            stats[f"helmet_images_{split}"] += 1
            stats[f"helmet_boxes_{split}"] += box_count
            if done % 500 == 0:
                log(f"  {done}/{len(jobs)} frames")
    stats["helmet_frame_errors"] = len(errors)
    if errors:
        log("First errors:", *errors[:5], sep="\n  ")
    if len(errors) > 0.05 * max(1, len(jobs)):
        raise RuntimeError(f"Too many HELMET frames failed ({len(errors)} of {len(jobs)})")
    stats["helmet_seconds"] = round(time.time() - started)
    return dict(sorted(stats.items()))


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stage1", type=Path, required=True, help="The stage1 folder of a prepared dataset")
    parser.add_argument("--limit-clips", type=int, default=None, help="Only this many clips, for a quick test")
    parser.add_argument("--workers", type=int, default=16)
    args = parser.parse_args()
    print(json.dumps(add_helmet_frames(args.stage1, workers=args.workers, limit_clips=args.limit_clips), indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile extra_datasets.py
#!/usr/bin/env python3
"""Adds the pictures of a YOLO-format dataset from Kaggle to the stage 1 training set.

Such datasets rarely have a box for every motorcycle with its riders, so only the
classes that do mean exactly that are kept (for example `TripleRiding`), and
pseudo_labels.py fills in the rest. The dataset's own train/valid/test split is kept.
"""
from __future__ import annotations

import re
import shutil
from collections import Counter
from pathlib import Path

from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
SPLITS = {"train": "train", "valid": "val", "val": "val", "test": "test"}


def key(name: str) -> str:
    return re.sub(r"[^a-z]", "", name.lower())


def read_names(yaml_path: Path) -> list[str]:
    """Class names from a data.yaml in list or mapping form."""
    text = yaml_path.read_text(encoding="utf-8")
    inline = re.search(r"^names:\s*\[(.*?)\]", text, re.M | re.S)
    if inline:
        return [n.strip().strip("'\"") for n in inline.group(1).split(",") if n.strip()]
    names = []
    block = re.search(r"^names:\s*\n((?:[ \t]+.*\n?)+)", text, re.M)
    for line in (block.group(1) if block else "").splitlines():
        match = re.match(r"\s+(?:-\s*|\d+:\s*)(.+)$", line)
        if match:
            names.append(match.group(1).strip().strip("'\""))
    if not names:
        raise ValueError(f"No class names in {yaml_path}")
    return names


def find_dataset(search_roots: list[Path], marker: str) -> Path | None:
    """The folder holding a data.yaml whose class names include `marker`."""
    for root in search_roots:
        if not root.exists():
            continue
        for yaml_path in sorted(root.rglob("data.yaml")):
            try:
                if key(marker) in {key(n) for n in read_names(yaml_path)}:
                    return yaml_path.parent
            except (ValueError, OSError):
                continue
    return None


def add_yolo_dataset(root: Path, stage1: Path, prefix: str, unit_classes: set[str], max_long_side: int = 1280) -> dict:
    """Copies images into stage1/images/<split>, keeping only boxes of the unit classes as class 0."""
    names = read_names(root / "data.yaml")
    keep = {index for index, name in enumerate(names) if key(name) in {key(c) for c in unit_classes}}
    stats: Counter = Counter()
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        split = SPLITS.get(folder.name.lower())
        if split is None or not (folder / "images").is_dir():
            continue
        (stage1 / "images" / split).mkdir(parents=True, exist_ok=True)
        (stage1 / "labels" / split).mkdir(parents=True, exist_ok=True)
        for image_path in sorted((folder / "images").iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            stem = f"{prefix}_{image_path.stem}"
            target = stage1 / "images" / split / f"{stem}.jpg"
            try:
                with Image.open(image_path) as opened:
                    if image_path.suffix.lower() in {".jpg", ".jpeg"} and max(opened.size) <= max_long_side:
                        shutil.copy2(image_path, target)
                    else:
                        image = opened.convert("RGB")
                        image.thumbnail((max_long_side, max_long_side))
                        image.save(target, quality=92)
            except OSError:
                stats["unreadable_images"] += 1
                continue
            lines = []
            label = folder / "labels" / f"{image_path.stem}.txt"
            if label.exists():
                for row in label.read_text(encoding="utf-8").splitlines():
                    parts = row.split()
                    if len(parts) == 5 and int(float(parts[0])) in keep:
                        lines.append("0 " + " ".join(parts[1:]))
            (stage1 / "labels" / split / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
            stats[f"{prefix}_images_{split}"] += 1
            stats[f"{prefix}_boxes_{split}"] += len(lines)
    if not any(k.startswith(f"{prefix}_images_") for k in stats):
        raise ValueError(f"No images found under {root}")
    return dict(sorted(stats.items()))


In [ ]:
%%writefile pseudo_labels.py
#!/usr/bin/env python3
"""Adds boxes for ridden motorcycles that the datasets left unlabelled.

Neither public dataset boxes every motorcycle: bikes far away, or partly outside
the picture, are often left out. Trained on that, the model learns to ignore
exactly those bikes. A large detector trained on COCO looks at every image here,
and where it sees a motorcycle with a rider that has no box, the union of the
bike and its riders becomes a new `motorcycle_unit` box. Parked bikes with
nobody on them stay unlabelled, as in the original data.

The rules below are plain functions so they can be tested without a GPU.
"""
from __future__ import annotations

from collections import Counter
from pathlib import Path

PERSON = 0
MOTORCYCLE = 3
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# A bike is only added when the detector is fairly sure about it.
MOTO_CONF = 0.40
PERSON_CONF = 0.30
# A detected bike counts as already labelled when half of it lies inside an existing box.
COVERED_SHARE = 0.5
# New boxes smaller than this share of the image's long side are left out.
MIN_HEIGHT_SHARE = 0.015
MIN_WIDTH_SHARE = 0.008

Box = tuple[float, float, float, float]  # x1, y1, x2, y2 in pixels


def area(box: Box) -> float:
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def intersection(a: Box, b: Box) -> float:
    return area((max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])))


def iou(a: Box, b: Box) -> float:
    shared = intersection(a, b)
    union = area(a) + area(b) - shared
    return shared / union if union > 0 else 0.0


def is_rider(person: Box, moto: Box) -> bool:
    """A person sitting on the bike, not someone walking past it."""
    mw, mh = moto[2] - moto[0], moto[3] - moto[1]
    ph = person[3] - person[1]
    if mw <= 0 or mh <= 0 or ph <= 0 or area(person) <= 0:
        return False
    if intersection(person, moto) / area(person) < 0.10:
        return False
    centre_x = (person[0] + person[2]) / 2
    if not moto[0] - 0.15 * mw <= centre_x <= moto[2] + 0.15 * mw:
        return False
    # The rider's body reaches into the bike and the head sits above or near its top.
    if person[3] < moto[1] + 0.25 * mh or person[1] > moto[1] + 0.35 * mh:
        return False
    return 0.3 * mh <= ph <= 2.5 * mh


def unit_box(moto: Box, persons: list[Box]) -> Box | None:
    riders = [p for p in persons if is_rider(p, moto)]
    if not riders:
        return None
    boxes = [moto, *riders]
    return (min(b[0] for b in boxes), min(b[1] for b in boxes), max(b[2] for b in boxes), max(b[3] for b in boxes))


def new_units(existing: list[Box], motos: list[tuple[Box, float]], persons: list[Box], width: int, height: int) -> list[Box]:
    """Boxes to add for ridden bikes that no existing box covers."""
    long_side = max(width, height)
    added: list[Box] = []
    for moto, _ in sorted(motos, key=lambda item: -item[1]):
        if any(intersection(moto, box) / max(area(moto), 1e-6) >= COVERED_SHARE for box in existing):
            continue
        unit = unit_box(moto, persons)
        if unit is None:
            continue
        unit = (max(0.0, unit[0]), max(0.0, unit[1]), min(float(width), unit[2]), min(float(height), unit[3]))
        if unit[3] - unit[1] < MIN_HEIGHT_SHARE * long_side or unit[2] - unit[0] < MIN_WIDTH_SHARE * long_side:
            continue
        if any(iou(unit, box) >= 0.3 for box in existing + added):
            continue
        added.append(unit)
    return added


def read_yolo(path: Path, width: int, height: int) -> list[Box]:
    if not path.exists():
        return []
    boxes = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cx, cy, w, h = (float(v) for v in parts[1:])
        boxes.append(((cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height))
    return boxes


def yolo_line(box: Box, width: int, height: int) -> str:
    x1, y1, x2, y2 = box
    return f"0 {(x1 + x2) / 2 / width:.6f} {(y1 + y2) / 2 / height:.6f} {(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}"


def drop_uncovered(stage1: Path, unboxed: dict[str, list[list[float]]]) -> dict:
    """Keeps an image whose dataset left a rider's bike unboxed only if the boxes now cover every such head.

    `unboxed` maps image names to the heads (cx, cy, w, h, 0..1) that had no bike box.
    """
    stats: Counter = Counter()
    for name, heads in unboxed.items():
        found = [(split, stage1 / "images" / split / name) for split in ("train", "val", "test")]
        found = [(split, path) for split, path in found if path.exists()]
        if not found:
            continue
        split, image = found[0]
        label = stage1 / "labels" / split / f"{Path(name).stem}.txt"
        boxes = []
        if label.exists():
            for line in label.read_text(encoding="utf-8").splitlines():
                parts = line.split()
                if len(parts) == 5:
                    boxes.append(tuple(float(v) for v in parts[1:]))
        covered = all(any(abs(hx - bx) <= bw / 2 and abs(hy - by) <= bh / 2 for bx, by, bw, bh in boxes) for hx, hy, *_ in heads)
        if covered:
            stats[f"recovered_{split}"] += 1
        else:
            image.unlink()
            label.unlink(missing_ok=True)
            stats[f"dropped_{split}"] += 1
    return dict(sorted(stats.items()))


def complete_labels(stage1: Path, splits=("train", "val", "test"), weights: str = "yolo11x.pt", imgsz: int = 1280,
                    device=0, batch: int = 8, log=print) -> tuple[dict, dict[str, list[Box]]]:
    """Adds pseudo boxes to stage1/labels/<split>. Returns stats and the added boxes per image path."""
    from ultralytics import YOLO

    model = YOLO(weights)
    stats: Counter = Counter()
    added_by_image: dict[str, list[Box]] = {}
    for split in splits:
        images = sorted(p for p in (stage1 / "images" / split).iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)
        log(f"Checking {len(images)} {split} images for unlabelled bikes")
        for start in range(0, len(images), 256):
            chunk = images[start : start + 256]
            results = model.predict([str(p) for p in chunk], imgsz=imgsz, conf=min(MOTO_CONF, PERSON_CONF),
                                    classes=[PERSON, MOTORCYCLE], device=device, batch=batch, verbose=False,
                                    stream=True, half=str(device) != "cpu")
            for image_path, result in zip(chunk, results):
                height, width = result.orig_shape
                boxes = result.boxes
                xyxy = boxes.xyxy.cpu().numpy().tolist()
                classes = boxes.cls.cpu().numpy().astype(int).tolist()
                scores = boxes.conf.cpu().numpy().tolist()
                motos = [(tuple(b), s) for b, c, s in zip(xyxy, classes, scores) if c == MOTORCYCLE and s >= MOTO_CONF]
                persons = [tuple(b) for b, c, s in zip(xyxy, classes, scores) if c == PERSON and s >= PERSON_CONF]
                label_path = stage1 / "labels" / split / f"{image_path.stem}.txt"
                existing = read_yolo(label_path, width, height)
                extra = new_units(existing, motos, persons, width, height)
                source = "helmet" if image_path.name.startswith("helmet_") else "original"
                stats[f"{source}_{split}_images"] += 1
                stats[f"{source}_{split}_boxes_before"] += len(existing)
                if extra:
                    lines = [line for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()] if label_path.exists() else []
                    lines += [yolo_line(box, width, height) for box in extra]
                    label_path.write_text("\n".join(lines), encoding="utf-8")
                    stats[f"{source}_{split}_boxes_added"] += len(extra)
                    stats[f"{source}_{split}_images_changed"] += 1
                    added_by_image[str(image_path)] = extra
    return dict(sorted(stats.items())), added_by_image


In [ ]:
import json
import shutil

import prepare_public_dataset

if PREPARED.exists():
    shutil.rmtree(PREPARED)
STAGE1 = PREPARED / "stage1"
if ROBOFLOW:
    # Images where a rider's bike has no box stay in for now; after the missing boxes are
    # filled in below, each is kept only if every such rider is inside a box.
    stats = prepare_public_dataset.prepare(RAW, PREPARED, keep_unboxed=True)
    print(json.dumps(stats, indent=2))
    waiting = stats.get("stage1_unboxed_bike_images_kept_for_now", 0)
    print(f"{waiting} images ({waiting / stats['images_total']:.0%}) have a bike without a box; the box filling below decides about them.")
    STAGE2_YAML = stats["stage2_yaml"]
else:
    prepare_public_dataset.make_dirs(STAGE1)
    stats = {"skipped": "no Roboflow key: stage 2 is not retrained"}
    STAGE2_YAML = None
STAGE1_YAML = str(prepare_public_dataset.write_yaml(STAGE1, ["motorcycle_unit"]))
LABEL_STATS = {"prepare": stats}


def subset(name, keep, label_root=STAGE1):
    """A test-only dataset with the stage 1 test images that `keep` accepts, for separate scores."""
    root = PREPARED / name
    for kind in ("images", "labels"):
        (root / kind / "test").mkdir(parents=True, exist_ok=True)
    count = 0
    for image in sorted((STAGE1 / "images" / "test").iterdir()):
        if not keep(image.name):
            continue
        shutil.copy2(image, root / "images" / "test" / image.name)
        label = label_root / "labels" / "test" / f"{image.stem}.txt"
        if label.exists():
            shutil.copy2(label, root / "labels" / "test" / label.name)
        count += 1
    lines = [f"path: {root.resolve().as_posix()}", "train: images/test", "val: images/test", "test: images/test", "names:", "  0: motorcycle_unit"]
    (root / "data.yaml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return str(root / "data.yaml"), count


ORIGINAL_TEST_YAML = None
if ROBOFLOW:
    # The first model was scored on these images with these labels (mAP50 0.698), so keep an untouched copy.
    FIRST_RULE = set((STAGE1 / prepare_public_dataset.FIRST_RULE_FILE).read_text(encoding="utf-8").splitlines())
    ORIGINAL_TEST_YAML, count = subset("stage1_original_test", lambda name: name in FIRST_RULE)
    print("Test images the first model was scored on:", count)


### Add frames from the HELMET dataset


In [ ]:
import helmet_dataset

try:
    LABEL_STATS["helmet"] = helmet_dataset.add_helmet_frames(
        STAGE1, every_train=HELMET_EVERY_TRAIN, every_eval=HELMET_EVERY_EVAL, long_side=1280, limit_clips=HELMET_LIMIT_CLIPS
    )
    print(json.dumps(LABEL_STATS["helmet"], indent=2))
except Exception as error:
    # The original dataset still trains a better model than before, so carry on, but say so loudly.
    LABEL_STATS["helmet"] = {"error": f"{type(error).__name__}: {error}"}
    print("WARNING: the HELMET frames could not be added, so this run uses the original dataset only.", error)


### Add Indian traffic pictures


In [ ]:
import extra_datasets

EXTRA = "devgurucodes/trafffic-violations-triple-riding-no-helmet-plate"
try:
    root = extra_datasets.find_dataset(EXTRA_SEARCH, "TripleRiding")
    if root is None:
        import kagglehub

        root = extra_datasets.find_dataset([Path(kagglehub.dataset_download(EXTRA))], "TripleRiding")
    if root is None:
        raise FileNotFoundError(f"{EXTRA} has no data.yaml with a TripleRiding class")
    LABEL_STATS["devguru"] = extra_datasets.add_yolo_dataset(root, STAGE1, "dg", {"TripleRiding"})
    print(json.dumps(LABEL_STATS["devguru"], indent=2))
except Exception as error:
    LABEL_STATS["devguru"] = {"error": f"{type(error).__name__}: {error}"}
    print("WARNING: the Indian traffic pictures could not be added, so this run goes on without them.", error)


### Box the ridden bikes the datasets missed


In [ ]:
import gc

import torch

import pseudo_labels

pseudo_stats, ADDED_BOXES = pseudo_labels.complete_labels(STAGE1, weights=LABEL_HELPER, imgsz=LABEL_HELPER_SIZE, device=DEVICE)
LABEL_STATS["pseudo_labels"] = pseudo_stats
print(json.dumps(pseudo_stats, indent=2))

unboxed_file = STAGE1 / prepare_public_dataset.UNBOXED_FILE
if unboxed_file.exists():
    LABEL_STATS["unboxed_bikes"] = pseudo_labels.drop_uncovered(STAGE1, json.loads(unboxed_file.read_text(encoding="utf-8")))
    print("Images with an unboxed bike:", json.dumps(LABEL_STATS["unboxed_bikes"]))
    ADDED_BOXES = {path: boxes for path, boxes in ADDED_BOXES.items() if Path(path).exists()}
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SOURCE_TESTS = {}
for prefix, label in (("helmet_", "helmet"), ("dg_", "indian_traffic")):
    yaml_path, count = subset(f"stage1_{label}_test", lambda name, p=prefix: name.startswith(p))
    print(label, "test images:", count)
    if count:
        SOURCE_TESTS[label] = yaml_path
if not any((STAGE1 / "images" / "train").iterdir()):
    raise RuntimeError("No training pictures at all: every dataset failed to load. See the warnings above.")
(SAVE_DIR / "label_stats.json").write_text(json.dumps(LABEL_STATS, indent=2))


In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw


def show(paths, title):
    figure, axes = plt.subplots(2, 3, figsize=(15, 8))
    for axis in axes.flat:
        axis.axis("off")
    for axis, path in zip(axes.flat, paths):
        image = Image.open(path).convert("RGB")
        draw = ImageDraw.Draw(image)
        label = STAGE1 / "labels" / Path(path).parent.name / f"{Path(path).stem}.txt"
        added = ADDED_BOXES.get(str(path), [])
        for box in pseudo_labels.read_yolo(label, image.width, image.height):
            is_added = any(max(abs(a - b) for a, b in zip(box, other)) < 2 for other in added)
            draw.rectangle(box, outline="yellow" if is_added else "red", width=max(2, image.width // 300))
        axis.imshow(image)
    figure.suptitle(title)
    plt.show()


rng = random.Random(SEED)
changed = sorted(ADDED_BOXES)
show(rng.sample(changed, min(6, len(changed))), "Red: the dataset's own boxes. Yellow: boxes added for bikes it missed")
for prefix, title in (("helmet_", "HELMET frames from Myanmar"), ("dg_", "Indian traffic pictures")):
    frames = sorted((STAGE1 / "images" / "train").glob(f"{prefix}*"))
    show(rng.sample(frames, min(6, len(frames))), f"{title}, with all their boxes")


## 3. Train

Stage 1 finds motorcycles. Stage 2 looks inside each one for heads with and without helmets.


In [ ]:
from ultralytics import YOLO


def train(name, data, imgsz, model, epochs, batch):
    run = RUNS / name
    done = run / "done.txt"
    last = run / "weights" / "last.pt"
    if done.exists():
        print(name, "is already trained.")
        return
    if last.exists():
        print("Carrying on from the last saved epoch of", name)
        try:
            YOLO(str(last)).train(resume=True)
        except AssertionError as error:
            # Ultralytics refuses to resume a run that had already finished.
            print(error)
    else:
        YOLO(model).train(
            data=data,
            imgsz=imgsz,
            epochs=epochs,
            patience=PATIENCE,
            batch=batch,
            device=DEVICE,
            workers=WORKERS,
            cache=CACHE,
            seed=SEED,
            deterministic=True,
            project=str(RUNS),
            name=name,
            exist_ok=True,
        )
    done.write_text("finished")


In [ ]:
train("stage1", STAGE1_YAML, STAGE1_SIZE, STAGE1_MODEL, STAGE1_EPOCHS, STAGE1_BATCH)


In [ ]:
if STAGE2_YAML:
    train("stage2", STAGE2_YAML, STAGE2_SIZE, STAGE2_MODEL, STAGE2_EPOCHS, STAGE2_BATCH)
else:
    print("Stage 2 is not retrained in this run: it needs the Roboflow dataset.")


## 4. Measure on unseen videos


In [ ]:
def evaluate(name, weights, data, imgsz):
    if not data:
        return None
    metrics = YOLO(str(RUNS / weights / "weights" / "best.pt")).val(data=data, split="test", imgsz=imgsz, device=DEVICE, project=str(VAL_DIR), name=name, exist_ok=True)
    rows = {}
    for index, class_id in enumerate(metrics.box.ap_class_index):
        precision, recall, map50, map50_95 = metrics.box.class_result(index)
        rows[metrics.names[int(class_id)]] = {
            "precision": round(float(precision), 3),
            "recall": round(float(recall), 3),
            "mAP50": round(float(map50), 3),
            "mAP50_95": round(float(map50_95), 3),
        }
    return rows


# Every test clip and video is one neither model saw during training.
results = {"stage1": evaluate("stage1", "stage1", STAGE1_YAML, STAGE1_SIZE)}
if ORIGINAL_TEST_YAML:
    results["stage1_original_test_original_labels"] = evaluate("stage1_original", "stage1", ORIGINAL_TEST_YAML, STAGE1_SIZE)
for label, yaml_path in SOURCE_TESTS.items():
    results[f"stage1_{label}_test"] = evaluate(f"stage1_{label}", "stage1", yaml_path, STAGE1_SIZE)
if STAGE2_YAML:
    results["stage2"] = evaluate("stage2", "stage2", STAGE2_YAML, STAGE2_SIZE)
print(json.dumps(results, indent=2))


def score(group, cls, key):
    return ((results.get(group) or {}).get(cls) or {}).get(key)


def target(label, value):
    print(f"Plan target, {label} >= 0.85:", "not measured" if value is None else ("met" if value >= 0.85 else "NOT met"), value)


print("Same test images as the first model (it scored mAP50 0.698):", score("stage1_original_test_original_labels", "motorcycle_unit", "mAP50"))
target("motorcycle mAP50", score("stage1", "motorcycle_unit", "mAP50"))
target("no_helmet precision", score("stage2", "no_helmet", "precision"))
target("no_helmet recall", score("stage2", "no_helmet", "recall"))
(SAVE_DIR / "metrics.json").write_text(json.dumps(results, indent=2))


## 5. Export for the phone and the website, and check it


In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image


def check_onnx_format(path, imgsz, class_count):
    # These are exactly the assumptions the app's and website's parsers make.
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    blank = np.random.default_rng(SEED).random((1, 3, imgsz, imgsz), dtype=np.float32)
    output = session.run(None, {session.get_inputs()[0].name: blank})[0]
    assert output.ndim == 3 and output.shape[1] == 4 + class_count, f"Unexpected output shape {output.shape}"
    scores = output[0, 4:]
    assert scores.min() >= 0 and scores.max() <= 1, "Class scores are not 0..1"
    assert output[0, :4].max() > 1.5, "Boxes are not in pixels"
    return output.shape


def letterbox(path, imgsz):
    # Black bars, centred, like the app and the website.
    image = Image.open(path).convert("RGB")
    scale = min(imgsz / image.width, imgsz / image.height)
    resized = image.resize((round(image.width * scale), round(image.height * scale)), Image.BILINEAR)
    canvas = Image.new("RGB", (imgsz, imgsz))
    canvas.paste(resized, ((imgsz - resized.width) // 2, (imgsz - resized.height) // 2))
    return (np.asarray(canvas, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None].copy()


def raw_output(result):
    # The PyTorch model returns its raw boxes alongside other tensors.
    if isinstance(result, dict):
        result = result.get("one2many", next(iter(result.values())))
    while isinstance(result, (list, tuple)):
        result = result[0]
    return result.detach().cpu().numpy()


def check_onnx_matches(best, onnx_path, images, imgsz):
    # The plan's check: on the same input, the exported model must give the same output as the trained one.
    # Raw outputs are compared, so a box scored right at a cut-off can't make a good export look bad.
    import torch

    model = YOLO(str(best)).model.float().eval()
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    score_gap = box_gap = 0.0
    for image in images:
        tensor = letterbox(image, imgsz)
        with torch.no_grad():
            expected = raw_output(model(torch.from_numpy(tensor)))
        got = session.run(None, {session.get_inputs()[0].name: tensor})[0]
        assert expected.shape == got.shape, f"Shapes differ: {expected.shape} vs {got.shape}"
        score_gap = max(score_gap, float(np.abs(expected[:, 4:] - got[:, 4:]).max()))
        box_gap = max(box_gap, float(np.abs(expected[:, :4] - got[:, :4]).max()))
    print(f"{best.parent.parent.name}: largest difference on {len(images)} test images: score {score_gap:.5f}, box {box_gap:.3f} px")
    assert score_gap < 0.01 and box_gap < 1.0, "The exported model disagrees with the trained model"


def export(name, imgsz, class_count, data_root):
    best = RUNS / name / "weights" / "best.pt"
    onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=imgsz, simplify=True, nms=False, device="cpu"))
    print(name, "output shape", check_onnx_format(onnx_path, imgsz, class_count))
    check_onnx_matches(best, onnx_path, sorted((data_root / "images" / "test").iterdir())[:20], imgsz)
    return onnx_path


stage1_onnx = export("stage1", STAGE1_SIZE, 1, STAGE1)
stage2_onnx = export("stage2", STAGE2_SIZE, len(STAGE2_NAMES), PREPARED / "stage2") if STAGE2_YAML else None


In [ ]:
import io
import json
import zipfile

from PIL import Image

manifest = {
    "version": VERSION,
    "stage1": {"file": "stage1.onnx", "input": STAGE1_SIZE, "classes": ["motorcycle_unit"]},
}
# Without the Roboflow dataset there is no new stage 2; the app keeps its current one.
if stage2_onnx:
    manifest["stage2"] = {"file": "stage2.onnx", "input": STAGE2_SIZE, "classes": STAGE2_NAMES}


def add_eval_sample(archive, dataset, prefix, count):
    # A few test images with labels, so the new model can be compared with the old one on a PC.
    images = sorted((dataset / "images" / "test").iterdir())
    for image in images[:: max(1, len(images) // count)][:count]:
        picture = Image.open(image).convert("RGB")
        picture.thumbnail((1280, 1280))
        buffer = io.BytesIO()
        picture.save(buffer, "JPEG", quality=90)
        archive.writestr(f"eval_sample/{prefix}/images/{image.stem}.jpg", buffer.getvalue())
        label = dataset / "labels" / "test" / f"{image.stem}.txt"
        archive.writestr(f"eval_sample/{prefix}/labels/{image.stem}.txt", label.read_text() if label.exists() else "")


output = SAVE_DIR / "helmetguard_models.zip"
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(stage1_onnx, "stage1.onnx")
    if stage2_onnx:
        archive.write(stage2_onnx, "stage2.onnx")
    archive.writestr("models.json", json.dumps(manifest, indent=2))
    archive.writestr("metrics.json", json.dumps(results, indent=2))
    archive.writestr("label_stats.json", json.dumps(LABEL_STATS, indent=2))
    if ORIGINAL_TEST_YAML:
        add_eval_sample(archive, PREPARED / "stage1_original_test", "original", 80)
    for label in SOURCE_TESTS:
        add_eval_sample(archive, PREPARED / f"stage1_{label}_test", label, 80)
print(f"{output.stat().st_size / 1e6:.1f} MB")
print("Done. Download helmetguard_models.zip from the Output section of this saved version.")


## Finished

Download **helmetguard_models.zip** from the **Output** section of this saved version. Keep it in your Downloads folder and tell Claude it's ready, so it can be installed in the app and the website.
